# Clase 118 — TensorFlow: tensores, variables, operaciones

Bajamos un nivel por debajo de Keras: trabajamos directamente con **tensores**
(`tf.Tensor`, inmutables) y **variables** (`tf.Variable`, mutables, la base de
los pesos). La API es *NumPy-like* pero corre en GPU y es diferenciable.

Requiere: `tensorflow` (≥ 2.x), `numpy`. Se ejecuta en Colab o en un entorno con TF instalado.

## 1. Tensores: creación y atributos

In [ ]:
import numpy as np
import tensorflow as tf
tf.random.set_seed(42)

t = tf.constant([[1.0, 2.0], [3.0, 4.0]])
ceros = tf.zeros((2, 3))
unos = tf.ones((2, 3))
aleatorio = tf.random.normal((2, 2), mean=0.0, stddev=1.0)

print("t =\n", t.numpy())
print("shape:", t.shape, "| dtype:", t.dtype)
print("aleatorio ~N(0,1):\n", aleatorio.numpy())

## 2. Operaciones: aritmética, `matmul`, `transpose`, `reduce_*`

In [ ]:
print("t + 10 =\n", (t + 10).numpy())
print("t * t (element-wise) =\n", (t * t).numpy())
print("matmul t @ t =\n", tf.matmul(t, t).numpy())
print("transpuesta =\n", tf.transpose(t).numpy())
print("suma por columnas (axis=0):", tf.reduce_sum(t, axis=0).numpy())
print("media global:", tf.reduce_mean(t).numpy())
print("máximo:", tf.reduce_max(t).numpy())

## 3. Broadcasting (reglas idénticas a NumPy)

In [ ]:
a = tf.constant([[1.0], [2.0], [3.0]])   # (3, 1)
b = tf.constant([10.0, 20.0, 30.0])      # (3,)
c = a + b                                # (3, 3) por broadcasting
print("a shape:", a.shape, "| b shape:", b.shape)
print("a + b shape:", c.shape)
print(c.numpy())

## 4. Indexado, slicing e inmutabilidad

In [ ]:
m = tf.constant([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("primera fila:", m[0].numpy())
print("última columna:", m[:, -1].numpy())
print("submatriz [:2, 1:]:\n", m[:2, 1:].numpy())

# los tensores son inmutables: no se puede m[mask] = 0; usamos tf.where
mascara = m > 5
m_clip = tf.where(mascara, tf.zeros_like(m), m)
print("valores > 5 puestos a 0:\n", m_clip.numpy())

## 5. `tf.Variable`: mutabilidad con `assign` / `assign_add` / `assign_sub`

In [ ]:
v = tf.Variable([1.0, 2.0, 3.0])
v.assign([4.0, 5.0, 6.0])        # reemplaza el contenido in-place
v.assign_add([1.0, 1.0, 1.0])    # v += 1
v.assign_sub([0.5, 0.5, 0.5])    # v -= 0.5
print("Variable final:", v.numpy())
print("¿es entrenable?:", v.trainable)

## 6. dtypes y `tf.cast`

In [ ]:
enteros = tf.constant([1, 2, 3])           # int32
flotantes = tf.constant([1.0, 2.0, 3.0])   # float32
# enteros + flotantes -> InvalidArgumentError por dtype mismatch
suma_ok = tf.cast(enteros, tf.float32) + flotantes
print("int32:", enteros.dtype, "| float32:", flotantes.dtype)
print("tras tf.cast:", suma_ok.numpy(), "| dtype:", suma_ok.dtype)
print("bool:", tf.constant([True, False]).dtype)

## 7. Conversión NumPy ⇄ TF

In [ ]:
arr = np.array([[1.0, 2.0], [3.0, 4.0]])
como_tf = tf.convert_to_tensor(arr, dtype=tf.float32)
de_vuelta = como_tf.numpy()
print("np -> tf:", type(como_tf).__name__)
print("tf -> np:", type(de_vuelta).__name__)
# muchas ops de TF aceptan arrays NumPy directamente
print("tf.reduce_sum sobre np array:", tf.reduce_sum(arr).numpy())

## 8. Homework: regresión lineal manual con `GradientTape`

In [ ]:
tf.random.set_seed(42)
X = tf.random.normal((100, 2))
w_true = tf.constant([[1.0], [2.0]])
y = X @ w_true + 3.0 + tf.random.normal((100, 1), stddev=0.1)

w = tf.Variable(tf.random.normal((2, 1)))
b = tf.Variable(0.0)
lr = 0.05

for paso in range(500):
    with tf.GradientTape() as tape:
        y_pred = X @ w + b
        loss = tf.reduce_mean(tf.square(y_pred - y))   # MSE
    gw, gb = tape.gradient(loss, [w, b])
    w.assign_sub(lr * gw)
    b.assign_sub(lr * gb)

print("w aprendido:", w.numpy().ravel(), "(esperado [1, 2])")
print("b aprendido:", float(b.numpy()), "(esperado 3)")
print("MSE final:", float(loss))

## Ejercicios

1. **Tensores básicos**: creá `t = tf.constant([[1., 2.], [3., 4.]])` e imprimí
   `t.shape`, `t.dtype` y `t.numpy()`.
2. **Operaciones**: verificá las shapes de `tf.matmul(t, t)`, `tf.transpose(t)`
   y `tf.reduce_sum(t, axis=0)`.
3. **Broadcasting**: con `a` de shape `(3, 1)` y `b` de shape `(3,)`, predecí y
   comprobá la shape de `a + b`.
4. **Variable y assign**: partiendo de `v = tf.Variable([1., 2., 3.])`, aplicá
   `assign`, `assign_add` y `assign_sub` y verificá el resultado.
5. **dtype mismatch**: sumá un tensor `int32` con uno `float32`; arreglá el error
   con `tf.cast`.

## Conclusiones

- `tf.Tensor` es **inmutable** (como `np.ndarray`); `tf.Variable` es **mutable** y es la base de los pesos.
- La API es NumPy-like: `matmul`, `reduce_*`, broadcasting y slicing funcionan igual, pero en GPU y diferenciable.
- Los dtypes deben coincidir: `tf.cast` resuelve los mismatches (`float32` es el default de ML).
- Para "editar" un tensor se usa `tf.where`, no asignación por índice.
- Con `GradientTape` + `Variable.assign_sub` se entrena a mano, sin Keras.